# 🏋️ PulseFit · Notebook 04
## Answer the CEO, then look under the hood

| 01 Set up | 02 Bronze | 03 Silver & Gold | 04 Answer the CEO | 05 Superpowers |
|:-:|:-:|:-:|:-:|:-:|
| ✅ done | ✅ done | ✅ done | 📍 **you are here** | ⬜ |

### Where we are
The gold layer is built **and tested**. Time to answer the question:

> **"Which members stopped visiting last quarter, and was it because of the price increase?"**

* **Part A**: answer it with SQL on the star schema, plus a few charts.
* **Part B**: find out *why* Snowflake answered so fast: micro-partitions and pruning, the result cache, and
  elastic compute.

## Step 0: Set the context
> If you see *"does not exist or not authorized"*, run notebooks **01** to **03** first.

In [ ]:
%%sql -r context
-- Every notebook starts by telling Snowflake WHO we are and WHERE we work
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE PULSEFIT_WH;
USE DATABASE PULSEFIT_DW;
USE SCHEMA GOLD;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_SCHEMA()    AS my_schema;

We'll also use a little Python for charts. This cell connects Python to the same Snowflake session:

In [ ]:
import time
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print("Connected as", session.get_current_role(), "using warehouse", session.get_current_warehouse())

# Part A: Answer the question
## Step 1: Warm-up: visits per club per month
This is the classic star-schema query from the lecture: **join the fact to the dimensions, then group**.

In [ ]:
%%sql -r visits_by_club_month
SELECT c.club_name,
       d.year_month,
       COUNT(*) AS visits
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_CLUB c ON c.club_key = f.club_key
JOIN GOLD.DIM_DATE d ON d.date_key = f.date_key
GROUP BY c.club_name, d.year_month
ORDER BY c.club_name, d.year_month;

The SQL result above is available in Python as `visits_by_club_month`. Let's chart it:

In [ ]:
df = visits_by_club_month.to_pandas()

alt.Chart(df, title="Visits per month, by club").mark_line(point=True).encode(
    x=alt.X("YEAR_MONTH:O", title="Month"),
    y=alt.Y("VISITS:Q", title="Visits"),
    color=alt.Color("CLUB_NAME:N", title="Club"),
    tooltip=["CLUB_NAME", "YEAR_MONTH", "VISITS"],
).properties(width=700, height=320)

👀 Colombo is the busiest club. Visits dip in **April to June** at every club. Is that the price increase, or
just the season? Let's find out.

## Step 2: Why history matters: Nimal's visits
Remember **Nimal Perera**, who moved from Kandy to Colombo on 1 March 2026? PulseFit reports visits by each
member's **home club**. How many January–February visits should be credited to each club?
* **Type 2**: join each visit to the member version **valid on the visit date** (`member_key`)
* **Type 1**: pretend we only kept the **current** version (overwrite)

In [ ]:
%%sql -r nimal_type1_vs_type2
-- SCD Type 2: the version that was true on the day of the visit
SELECT 'Type 2 (history kept)' AS approach,
       m.home_club,
       COUNT(*)                AS jan_feb_visits
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_MEMBER m ON m.member_key = f.member_key
JOIN GOLD.DIM_DATE   d ON d.date_key   = f.date_key
WHERE m.member_id = 1001
  AND d.full_date BETWEEN '2026-01-01' AND '2026-02-28'
GROUP BY m.home_club

UNION ALL

-- SCD Type 1: only today's version exists
SELECT 'Type 1 (overwritten)',
       today.home_club,
       COUNT(*)
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_MEMBER m     ON m.member_key = f.member_key
JOIN GOLD.DIM_MEMBER today ON today.member_id = m.member_id AND today.is_current
JOIN GOLD.DIM_DATE   d     ON d.date_key = f.date_key
WHERE m.member_id = 1001
  AND d.full_date BETWEEN '2026-01-01' AND '2026-02-28'
GROUP BY today.home_club;

👀 **Kandy 24 · Colombo 0** with Type 2, but **Kandy 0 · Colombo 24** with Type 1. That's exactly the lecture
slide, now built from real data. Same visits, same kind of SQL, different answers. Only Type 2 tells the truth about
January and February.

The CEO's question has the same trap: *"which plan was a member on **before** the price rise?"* needs history too.

## Step 3: The CEO's question
First, turn the vague question into a precise definition:
* **Active in Q1**: visited at least once in **Q1 2026** (January to March).
* **Stopped visiting in Q2**: active in Q1, but **zero visits in Q2 2026** (April to June).
* Group members by the **plan they were on at the end of Q1** (31 March), using a point-in-time lookup on the SCD
  Type 2 history, and show each plan's price **before and after** 1 April.

In [ ]:
%%sql -r lapse_by_plan
WITH visits AS (
    SELECT m.member_id, d.year_quarter
    FROM GOLD.FCT_CHECKIN f
    JOIN GOLD.DIM_MEMBER m ON m.member_key = f.member_key
    JOIN GOLD.DIM_DATE   d ON d.date_key   = f.date_key
),
active_q1 AS (SELECT DISTINCT member_id FROM visits WHERE year_quarter = '2026-Q1'),
active_q2 AS (SELECT DISTINCT member_id FROM visits WHERE year_quarter = '2026-Q2'),
plan_on_31_march AS (
    SELECT member_id, plan_code
    FROM GOLD.DIM_MEMBER
    WHERE '2026-03-31'::DATE >= valid_from AND '2026-03-31'::DATE < valid_to
)
SELECT p.plan_code,
       MAX(fee_q1.monthly_fee_lkr)                     AS fee_in_q1_lkr,
       MAX(fee_q2.monthly_fee_lkr)                     AS fee_in_q2_lkr,
       COUNT(*)                                        AS active_in_q1,
       COUNT_IF(q2.member_id IS NULL)                  AS stopped_visiting_in_q2,
       ROUND(100 * stopped_visiting_in_q2 / active_in_q1, 1) AS stopped_pct
FROM active_q1 q1
JOIN plan_on_31_march p ON p.member_id = q1.member_id
LEFT JOIN active_q2 q2  ON q2.member_id = q1.member_id
JOIN GOLD.DIM_PLAN fee_q1
  ON fee_q1.plan_code = p.plan_code AND '2026-03-31'::DATE >= fee_q1.valid_from AND '2026-03-31'::DATE < fee_q1.valid_to
JOIN GOLD.DIM_PLAN fee_q2
  ON fee_q2.plan_code = p.plan_code AND '2026-04-01'::DATE >= fee_q2.valid_from AND '2026-04-01'::DATE < fee_q2.valid_to
GROUP BY p.plan_code
ORDER BY stopped_pct DESC;

In [ ]:
df = lapse_by_plan.to_pandas()
df["PRICE_CHANGE"] = df.apply(
    lambda r: "price went up" if r["FEE_IN_Q2_LKR"] > r["FEE_IN_Q1_LKR"] else "no price change", axis=1)

alt.Chart(df, title="% of Q1 members who stopped visiting in Q2 2026, by plan").mark_bar().encode(
    x=alt.X("PLAN_CODE:N", sort="-y", title="Plan on 31 March 2026"),
    y=alt.Y("STOPPED_PCT:Q", title="% stopped visiting"),
    color=alt.Color("PRICE_CHANGE:N", title="",
                    scale=alt.Scale(domain=["price went up", "no price change"], range=["#d62728", "#9e9e9e"])),
    tooltip=["PLAN_CODE", "FEE_IN_Q1_LKR", "FEE_IN_Q2_LKR", "ACTIVE_IN_Q1", "STOPPED_VISITING_IN_Q2", "STOPPED_PCT"],
).properties(width=500, height=320)

## Step 4: When exactly did they stop?
If the price increase is the cause, Standard members should drop away **right around the announcement
(15 March) and the new price (1 April)**, while the other plans stay steady. Let's count active members per week,
shown as an index (average of January–February = 100) so plans of different sizes can share one chart:

In [ ]:
%%sql -r weekly_active_by_plan
WITH weekly AS (
    SELECT d.week_start,
           m.plan_code,
           COUNT(DISTINCT m.member_id) AS active_members
    FROM GOLD.FCT_CHECKIN f
    JOIN GOLD.DIM_DATE   d ON d.date_key   = f.date_key
    JOIN GOLD.DIM_MEMBER m ON m.member_key = f.member_key
    WHERE d.full_date BETWEEN '2025-10-06' AND '2026-06-28'
    GROUP BY d.week_start, m.plan_code
)
SELECT week_start,
       plan_code,
       active_members,
       ROUND(100 * active_members
             / AVG(IFF(week_start BETWEEN '2026-01-05' AND '2026-02-23', active_members, NULL))
                   OVER (PARTITION BY plan_code), 1) AS index_vs_jan_feb
FROM weekly
ORDER BY week_start, plan_code;

In [ ]:
df = weekly_active_by_plan.to_pandas()
df["WEEK_START"] = pd.to_datetime(df["WEEK_START"])

lines = alt.Chart(df, title="Active members per week (Jan–Feb average = 100)").mark_line().encode(
    x=alt.X("WEEK_START:T", title="Week"),
    y=alt.Y("INDEX_VS_JAN_FEB:Q", title="Index", scale=alt.Scale(zero=False)),
    color=alt.Color("PLAN_CODE:N", title="Plan"),
    tooltip=["WEEK_START", "PLAN_CODE", "ACTIVE_MEMBERS", "INDEX_VS_JAN_FEB"],
)
events = pd.DataFrame({"date": pd.to_datetime(["2026-03-15", "2026-04-01"]),
                       "label": ["price rise announced", "new price starts"]})
rules = alt.Chart(events).mark_rule(strokeDash=[4, 4], color="black").encode(x="date:T")
labels = alt.Chart(events).mark_text(align="left", dx=4, dy=-120, fontSize=11).encode(x="date:T", text="label")

(lines + rules + labels).properties(width=750, height=340)

👀 Basic, Premium and Student stay roughly level (they even grow a little as new members join). **Standard drops by
about a third within two weeks of the announcement** and never recovers.

## Step 5: The answer, in plain words
This cell writes the answer for the CEO, straight from the query results:

In [ ]:
df = lapse_by_plan.to_pandas()
std = df[df["PLAN_CODE"] == "STANDARD"].iloc[0]
others = df[df["PLAN_CODE"] != "STANDARD"]
others_pct = 100 * others["STOPPED_VISITING_IN_Q2"].sum() / others["ACTIVE_IN_Q1"].sum()
rise_pct = 100 * (std["FEE_IN_Q2_LKR"] - std["FEE_IN_Q1_LKR"]) / std["FEE_IN_Q1_LKR"]

print("📣 ANSWER FOR THE CEO")
print("=" * 70)
print(f"• {int(std['STOPPED_VISITING_IN_Q2'])} of {int(std['ACTIVE_IN_Q1'])} Standard members "
      f"({std['STOPPED_PCT']}%) visited in Q1 but not once in Q2 2026.")
print(f"• That followed the Standard price rise from LKR {int(std['FEE_IN_Q1_LKR']):,} "
      f"to LKR {int(std['FEE_IN_Q2_LKR']):,} (+{rise_pct:.1f}%), announced on 15 March.")
print(f"• On the plans whose price did NOT change, only {others_pct:.1f}% stopped visiting.")
print(f"• Standard members started dropping away right after the announcement; other plans stayed steady.")
print("=" * 70)
print("Conclusion: the evidence strongly points to the price increase.")

### 🤔 Careful: correlation is not causation
The timing and the comparison with other plans make a strong case, but a good analyst checks for other
explanations. For example: **did one club cause this** (a renovation, a new competitor next door)?
If so, the drop would be concentrated in one club. Let's check:

In [ ]:
%%sql -r stopped_by_club_and_plan
WITH visits AS (
    SELECT m.member_id, d.year_quarter
    FROM GOLD.FCT_CHECKIN f
    JOIN GOLD.DIM_MEMBER m ON m.member_key = f.member_key
    JOIN GOLD.DIM_DATE   d ON d.date_key   = f.date_key
),
active_q1 AS (SELECT DISTINCT member_id FROM visits WHERE year_quarter = '2026-Q1'),
active_q2 AS (SELECT DISTINCT member_id FROM visits WHERE year_quarter = '2026-Q2'),
member_on_31_march AS (
    SELECT member_id, plan_code, home_club
    FROM GOLD.DIM_MEMBER
    WHERE '2026-03-31'::DATE >= valid_from AND '2026-03-31'::DATE < valid_to
)
SELECT m.home_club,
       ROUND(100 * COUNT_IF(m.plan_code = 'STANDARD' AND q2.member_id IS NULL)
                 / NULLIF(COUNT_IF(m.plan_code = 'STANDARD'), 0), 1)  AS standard_stopped_pct,
       ROUND(100 * COUNT_IF(m.plan_code <> 'STANDARD' AND q2.member_id IS NULL)
                 / NULLIF(COUNT_IF(m.plan_code <> 'STANDARD'), 0), 1) AS other_plans_stopped_pct
FROM active_q1 q1
JOIN member_on_31_march m ON m.member_id = q1.member_id
LEFT JOIN active_q2 q2    ON q2.member_id = q1.member_id
GROUP BY m.home_club
ORDER BY m.home_club;

👀 The pattern is the same at **every club**: Standard members stopped far more often than everyone else. So it
isn't a local problem. A sensible next step for PulseFit would be to ask some of those members why they left,
but the data has done its job. **Question answered.** 🎉

# Part B: Under the hood: why was that so fast?
PulseFit is small (about 200,000 visits). To *feel* Snowflake's engine we need something big. Every Snowflake account
comes with **`SNOWFLAKE_SAMPLE_DATA`**, shared by Snowflake (more on sharing in notebook 05). Its
`TPCH_SF100.LINEITEM` table has **600 million rows**.

Three ideas from the lecture make Snowflake fast **without indexes or tuning**:

| Idea | In one sentence |
|---|---|
| **Micro-partitions + pruning** | tables are stored in small compressed chunks with min/max metadata, so Snowflake skips chunks that can't match your filter |
| **Result cache** | run the exact same query again on unchanged data and the answer comes back instantly, without using a warehouse |
| **Elastic compute** | need more power? resize the warehouse in seconds, then size it back down |

> ⚠️ Part B times real queries, so your numbers will differ slightly from your neighbour's. That's expected.

## Step 6: How big is the table?

In [ ]:
%%sql -r lineitem_size
SELECT table_name,
       row_count,
       ROUND(bytes / POWER(1024, 3), 1) AS size_gb
FROM SNOWFLAKE_SAMPLE_DATA.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'TPCH_SF100'
  AND table_name   = 'LINEITEM';

## Step 7: Pruning: skip what you don't need
We ask for the items shipped on **one single day** out of seven years of data. The cell runs the query, then asks
Snowflake how many micro-partitions it actually had to read. (We switch the result cache off so Snowflake really
does the work.)

In [ ]:
session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()

query = '''
    SELECT COUNT(*) AS line_items, SUM(l_extendedprice) AS revenue
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF100.LINEITEM
    WHERE l_shipdate = '1995-03-15'
'''
t0 = time.time()
job = session.sql(query).collect_nowait()
rows = job.result()
seconds = time.time() - t0

stats = session.sql(f'''
    SELECT operator_statistics:pruning:partitions_scanned::INT AS scanned,
           operator_statistics:pruning:partitions_total::INT   AS total
    FROM TABLE(GET_QUERY_OPERATOR_STATS('{job.query_id}'))
    WHERE operator_type = 'TableScan'
''').collect()[0]

print(f"Found {rows[0]['LINE_ITEMS']:,} line items in {seconds:.1f} s")
print(f"Micro-partitions read: {stats['SCANNED']:,} of {stats['TOTAL']:,} "
      f"→ skipped {100 * (1 - stats['SCANNED'] / stats['TOTAL']):.1f}% of the table without reading it")

👀 Snowflake read only a **tiny fraction** of the table. For every micro-partition it keeps the min and max of
each column, so it knew without looking that most chunks contain no rows for 15 March 1995. Nobody designed an
index for this: it's automatic (lecture: *"Storage: micro-partitions and pruning"*).

## Step 8: Elastic compute and the result cache
Now a heavy query: the famous **TPC-H Query 1**, which summarises almost all 600 million rows. We will:
1. run it on our **X-Small** warehouse,
2. **resize to Medium** (4× the compute) and run it again,
3. resize back to X-Small,
4. switch the result cache back on and run it twice more.

This cell takes about 30 seconds.

In [ ]:
tpch_q1 = '''
    SELECT l_returnflag, l_linestatus,
           SUM(l_quantity)                          AS sum_qty,
           SUM(l_extendedprice)                     AS sum_base_price,
           SUM(l_extendedprice * (1 - l_discount))  AS sum_disc_price,
           AVG(l_quantity)                          AS avg_qty,
           COUNT(*)                                 AS count_order
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF100.LINEITEM
    WHERE l_shipdate <= DATEADD(day, -90, '1998-12-01'::DATE)
    GROUP BY l_returnflag, l_linestatus
    ORDER BY l_returnflag, l_linestatus
'''
timings = []

def timed_run(label):
    t0 = time.time()
    session.sql(tpch_q1).collect()
    timings.append({"run": label, "seconds": round(time.time() - t0, 2)})

try:
    session.sql("ALTER SESSION SET USE_CACHED_RESULT = FALSE").collect()
    timed_run("1. X-Small (1 credit/hour)")

    session.sql("ALTER WAREHOUSE PULSEFIT_WH SET WAREHOUSE_SIZE = 'MEDIUM'").collect()
    timed_run("2. Medium (4 credits/hour)")
finally:
    # always size back down, even if something above fails
    session.sql("ALTER WAREHOUSE PULSEFIT_WH SET WAREHOUSE_SIZE = 'XSMALL'").collect()
    session.sql("ALTER SESSION SET USE_CACHED_RESULT = TRUE").collect()

timed_run("3. X-Small, cache on (1st run fills the cache)")
timed_run("4. X-Small, same query again (from the result cache)")

results = pd.DataFrame(timings)
print(results.to_string(index=False))

alt.Chart(results, title="TPC-H Query 1 on 600 million rows").mark_bar().encode(
    x=alt.X("seconds:Q", title="Seconds"),
    y=alt.Y("run:N", title="", sort=None),
    tooltip=["run", "seconds"],
).properties(width=600, height=200)

👀 **What to notice**
* **Medium was much faster** than X-Small. It costs 4× more per second but runs for far fewer seconds, so the cost
  of the query is similar, and you wait much less. Resizing took seconds, and we sized back down straight after.
* **Run 4 came back almost instantly.** Same query, same data: Snowflake returned the stored result without using
  the warehouse at all. That costs nothing.
* This is the separation of storage and compute in action: the 600 million rows never moved; we only changed
  the compute pointed at them.

## ✅ What you did
* Answered the CEO's question with **star-schema SQL** and charts: Standard members stopped visiting at roughly
  **ten times** the rate of other plans, right after the price increase.
* Saw **SCD Type 1 vs Type 2** change an answer, using the exact example from the lecture.
* Checked an alternative explanation (one club) instead of jumping to conclusions.
* Saw **pruning**, the **result cache** and **elastic compute** on 600 million rows.

## 🤔 Think about it
1. Our definition of "stopped visiting" was *zero visits in Q2*. A member who came once in April and never again
   is **not** counted. Is that a good definition? What would you suggest instead?
2. The Medium warehouse cost 4× more per second. When is scaling **up** (bigger warehouse) the right answer, and
   when is scaling **out** (more clusters of the same size) better?

## ➡️ Next
Open **`05_Snowflake_Superpowers`**. Things are about to go wrong (on purpose).